<a href="https://colab.research.google.com/github/khanhngoo/DataVisualization_Project1_Team_23and4/blob/main/Project1_MLProcess_DataVisualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA & PREPROCESSING PIPELINE
# ==========================================
url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
df = pd.read_csv(url)

for col in ['species', 'island', 'sex']:
    df[col] = df[col].astype('category')

num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])
df[['sex']] = SimpleImputer(strategy='most_frequent').fit_transform(df[['sex']])
df = df[df['body_mass_g'] > 0]

ml_features = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(ml_features)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_features)
df['PCA_1'] = pca_result[:, 0]
df['PCA_2'] = pca_result[:, 1]

# ==========================================
# 2. PLOT GENERATION LOGIC
# ==========================================
dropdown_options = [
    'Flipper Length (mm)', 'Bill Length (mm)', 'Bill Depth (mm)',
    'Body Mass (g)', 'Principal Component 1 (PCA)', 'Principal Component 2 (PCA)'
]

def update_plot(x_label, y_label, color_mode, k_value):
    axis_map = {
        'Flipper Length (mm)': 'flipper_length_mm',
        'Bill Length (mm)': 'bill_length_mm',
        'Bill Depth (mm)': 'bill_depth_mm',
        'Body Mass (g)': 'body_mass_g',
        'Principal Component 1 (PCA)': 'PCA_1',
        'Principal Component 2 (PCA)': 'PCA_2'
    }

    x_col = axis_map[x_label]
    y_col = axis_map[y_label]
    df_plot = df.copy()

    if color_mode == 'Actual Species (Ground Truth)':
        target_col = 'species'
        color_map = {'Adelie': '#3B5BBE', 'Chinstrap': '#E14BA1', 'Gentoo': '#F9C555'}
        title = f"Biological Ground Truth: {x_label} vs {y_label} (Split by Sex)"
    else:
        kmeans = KMeans(n_clusters=int(k_value), random_state=42, n_init=10)
        df_plot['Cluster'] = kmeans.fit_predict(scaled_features).astype(str)
        target_col = 'Cluster'
        color_map = {'0': '#00A8E8', '1': '#D90429', '2': '#8338EC', '3': '#F9C555', '4': '#2EC4B6'}
        title = f"Unsupervised ML: K-Means (k={k_value}) (Split by Sex)"

    # VISUAL UPDATE: Added facet_col='sex' to split the charts
    fig = px.scatter(
        df_plot, x=x_col, y=y_col, color=target_col,
        symbol='island',
        facet_col='sex',
        color_discrete_map=color_map,
        hover_data=['species', 'island', 'year', 'sex', 'body_mass_g'],
        title=f"<b>{title}</b>", labels={x_col: x_label, y_col: y_label}
    )

    fig.update_traces(marker=dict(size=8, opacity=0.8, line=dict(width=0.5, color='#505050')))

    fig.update_layout(
        plot_bgcolor='white', font=dict(family="sans-serif", color="#505050"),
        xaxis=dict(showgrid=False, showline=True, linecolor='#D3D3D3', zeroline=False),
        yaxis=dict(showgrid=True, gridcolor='#EBEBEB', showline=True, linecolor='#D3D3D3', zeroline=False),
        height=550,
        legend_title_text='Classification'
    )
    # Clean up the subplot titles
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].title()))

    return fig

# ==========================================
# 3. GRADIO DASHBOARD UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as dashboard:
    gr.Markdown("# 🐧 Palmer Penguins InfoVis Dashboard")
    gr.Markdown("Explore biological ground truth vs. unsupervised machine learning using biological measurements encoding.")

    with gr.Row():
        with gr.Column(scale=1):
            x_axis = gr.Dropdown(choices=dropdown_options, value='Principal Component 1 (PCA)', label="X-Axis")
            y_axis = gr.Dropdown(choices=dropdown_options, value='Principal Component 2 (PCA)', label="Y-Axis")

            gr.Markdown("---")
            color_by = gr.Radio(choices=['Actual Species (Ground Truth)', 'K-Means Algorithm'], value='Actual Species (Ground Truth)', label="Color Points By:")
            k_slider = gr.Slider(minimum=2, maximum=5, step=1, value=3, label="Number of Clusters (k)")

        with gr.Column(scale=3):
            plot_output = gr.Plot()

    inputs = [x_axis, y_axis, color_by, k_slider]
    x_axis.change(fn=update_plot, inputs=inputs, outputs=plot_output)
    y_axis.change(fn=update_plot, inputs=inputs, outputs=plot_output)
    color_by.change(fn=update_plot, inputs=inputs, outputs=plot_output)
    k_slider.change(fn=update_plot, inputs=inputs, outputs=plot_output)
    dashboard.load(fn=update_plot, inputs=inputs, outputs=plot_output)

dashboard.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
